In [38]:
import numpy as np
import  pandas as pd 
import networkx as nx
import matplotlib.pyplot as plt
import geopandas as gpd
from geopy.distance import geodesic
import contextily as ctx
from sklearn.preprocessing import MinMaxScaler



### Load data

In [39]:
houses =pd.read_csv("minimal_example/synthetic_data/houses_v2.csv")
regions =pd.read_csv("minimal_example/synthetic_data/regions.csv")
schools =pd.read_csv("minimal_example/synthetic_data/schools_v2.csv")
stations =pd.read_csv("minimal_example/synthetic_data/stations_v2.csv")


In [40]:
houses.head()

,property_id,price,rooms,bathrooms,year_renovated,eco_score,heating_type,exterior_material,lat,lon,surface_total,year_renovated.1,eco_score.1,heating_type.1,exterior_material.1
0,H0,5.247241e+08,3,1,2000,75.0,gas,brick,4.620924,-74.129537,199.035382,2000,75.0,gas,brick
1,H1,8.704286e+08,2,1,2000,75.0,gas,brick,4.643822,-74.142194,106.877704,2000,75.0,gas,brick
2,H2,7.391964e+08,3,2,2000,75.0,gas,brick,4.654954,-74.036134,281.864080,2000,75.0,gas,brick
3,H3,6.591951e+08,5,2,2000,75.0,gas,brick,4.668410,-74.034124,151.755996,2000,75.0,gas,brick
4,H4,3.936112e+08,5,1,2000,75.0,gas,brick,4.717776,-74.052992,232.504457,2000,75.0,gas,brick


### Add nodes

In [41]:
G = nx.Graph()

In [42]:
for _, row in houses.iterrows():
    G.add_node(
     
     row['property_id'],
     type = 'house',
     price = row['price'],
     rooms = row['rooms'],
     bathrooms = row['bathrooms'],
     surface_total = row['surface_total'],
     lat = row['lat'],
     lon = row['lon']
     
        
    )

In [43]:
for _, row in stations.iterrows():
    G.add_node(
     
     row['poi_id'],
     type = 'station',
     daily_traffic = row['daily_traffic'],
     connections = row['connections'],
     lat = row['lat'],
     lon = row['lon']
     
        
    )

In [44]:
for _, row in schools.iterrows():
    G.add_node(
     row['poi_id'],
     type = 'school',
     ranking = row['ranking'],
     students = row['students'],
     lat = row['lat'],
     lon = row['lon']
     
        
    )

### Add edges

In [45]:
def add_edges(G, df1, df2):
    df1_id = df1.columns[df1.columns.str.endswith("_id")][0]
    df2_id = df2.columns[df2.columns.str.endswith("_id")][0]
    
    for _, row in df1.iterrows():
        coords_1 = (row['lat'], row['lon'])
        
        for _, rowa in df2.iterrows():
            coords_2 = (rowa['lat'], rowa['lon'])
            
            distance = geodesic(coords_1, coords_2).km
            
            if distance < 5:  # Connect if within 5 km
                
                G.add_edge(
                    row[df1_id], rowa[df2_id], weight= 1/distance)
    return G
            

In [46]:
G = add_edges(G, houses, stations)
G = add_edges(G, houses, schools)
G = add_edges(G, schools, stations)

In [47]:
G.edges()

EdgeView([('H0', 'T2'), ('H0', 'T4'), ('H0', 'S4'), ('H1', 'T2'), ('H1', 'T4'), ('H1', 'S4'), ('H3', 'T3'), ('H3', 'S3'), ('H4', 'T3'), ('H4', 'T5'), ('H4', 'S1'), ('H4', 'S3'), ('H5', 'T2'), ('H5', 'T4'), ('H5', 'S2'), ('H5', 'S4'), ('H6', 'T2'), ('H6', 'T4'), ('H6', 'S2'), ('H6', 'S4'), ('H7', 'T1'), ('H7', 'T3'), ('H7', 'T5'), ('H7', 'S1'), ('H7', 'S2'), ('H7', 'S3'), ('H8', 'T4'), ('T1', 'S1'), ('T1', 'S2'), ('T1', 'S3'), ('T2', 'S1'), ('T2', 'S2'), ('T2', 'S4'), ('T3', 'S1'), ('T3', 'S3'), ('T4', 'S2'), ('T4', 'S4'), ('T5', 'S3')])

### X feature Matrixes

In [48]:
def build_X_features(df, target=None):
    df1 = df.copy()
    
    # 1. Drop ID + lat/lon
    id_cols = [c for c in df1.columns if c.endswith("_id")]
    df1 = df1.drop(columns=id_cols + ["lat", "lon"], errors="ignore")
    
    if target is not None and target in df1.columns:
        df1 = df1.drop(columns=target, errors="ignore")

    # 2. Separate numeric + categorical
    numeric_cols = df1.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = df1.select_dtypes(include=["object", "category", "bool"]).columns

    # 3. Impute missing values
    df1[numeric_cols] = df1[numeric_cols].fillna(df1[numeric_cols].median())
    df1[cat_cols] = df1[cat_cols].fillna("Unknown")

    # 4. One-hot encode categoricals
    df1 = pd.get_dummies(df1, columns=cat_cols, drop_first=False)

    # 5. Scale numeric columns (per type!)
    scaler = MinMaxScaler()
    df1[numeric_cols] = scaler.fit_transform(df1[numeric_cols])

    return df1


In [49]:
house_X = build_X_features(houses, 'price')
school_X = build_X_features(schools)
station_X = build_X_features(stations)